# ✈️ S6E10 Route-ID Tri-GBDT Master Pipeline
### Kaggle Playground Series Season 6 Episode 10 · Predicting Airline Passenger Satisfaction
**Evaluation Metric:** ROC-AUC

**Key Architectural Innovations (from NotebookLM & Community Synthesis):**
1. **Route-ID Engineering:** Bayesian Smoothed In-Fold Target Encoding & Frequency Counts on `Flight Distance` (~3,500 unique repeating routes).
2. **Domain Satisfaction Physics:** Inflight comfort, digital experience, service quality, ground services, and asymmetric pain-point counters (ratings $\le 2$).
3. **Flight Dynamics:** Total delay, delay recovery (`Arrival Delay - Departure Delay`), and zero-delay indicators.
4. **Tri-GBDT 5-Fold Stratified CV:** Identical fold splits across LightGBM, CatBoost, and XGBoost.
5. **Nelder-Mead Rank Percentile Blending:** Softmax convex combination maximizing OOF ROC-AUC.


In [ ]:
import os
import gc
import sys
import json
import warnings
from pathlib import Path
from datetime import datetime
from typing import List, Tuple

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import roc_auc_score
from scipy.stats import rankdata
from scipy.optimize import minimize

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier

warnings.filterwarnings("ignore")

def find_data():
    candidates = [
        Path("/kaggle/input/competitions/playground-series-s6e10"),
        Path("/kaggle/input/playground-series-s6e10"),
        Path("./data/raw"),
        Path("../data/raw")
    ]
    for c in candidates:
        if (c / "train.csv").exists():
            print(f"📂 Found dataset at: {c}")
            return c / "train.csv", c / "test.csv", c / "sample_submission.csv"
    for root, dirs, files in os.walk("/kaggle/input"):
        if "train.csv" in files:
            p = Path(root)
            print(f"📂 Found dataset at: {p}")
            return p / "train.csv", p / "test.csv", p / "sample_submission.csv"
    raise FileNotFoundError("Could not find competition data!")

train_path, test_path, sub_path = find_data()
train_raw = pd.read_csv(train_path)
test_raw = pd.read_csv(test_path)
sub = pd.read_csv(sub_path)

if train_raw["satisfaction"].dtype == bool:
    train_raw["satisfaction"] = train_raw["satisfaction"].astype(int)
elif train_raw["satisfaction"].dtype == object:
    train_raw["satisfaction"] = train_raw["satisfaction"].apply(lambda x: 1 if str(x).lower() in ["1", "true", "satisfied"] else 0)

print(f"Train: {train_raw.shape} | Test: {test_raw.shape}")
print(f"Satisfaction Rate: {train_raw["satisfaction"].mean():.4%}")


## 2. Domain Physics & Base Feature Engineering
Engineering domain satisfaction indices, row-wise rating stats, asymmetric friction/pain points (ratings $\le 2$), delay recovery, and route frequency counts.


In [ ]:
SURVEY_FEATURES = [
    "Inflight wifi service", "Departure/Arrival time convenient", "Ease of Online booking",
    "Gate location", "Food and drink", "Online boarding", "Seat comfort",
    "Inflight entertainment", "On-board service", "Leg room service", "Checkin service", "Cleanliness"
]

def engineer_base_features(train_df, test_df):
    tr = train_df.copy()
    te = test_df.copy()

    # Route frequency count (leak-free across full distance spectrum)
    full_dist = pd.concat([tr["Flight Distance"], te["Flight Distance"]], axis=0)
    dist_freq = full_dist.value_counts().to_dict()

    for df in [tr, te]:
        # 1. Missing arrival delay handling & delay dynamics
        df["is_missing_arrival_delay"] = df["Arrival Delay in Minutes"].isnull().astype(np.int8)
        df["Arrival Delay in Minutes"] = df["Arrival Delay in Minutes"].fillna(df["Departure Delay in Minutes"])
        df["total_delay"] = df["Departure Delay in Minutes"] + df["Arrival Delay in Minutes"]
        df["delay_recovery"] = df["Arrival Delay in Minutes"] - df["Departure Delay in Minutes"]
        df["delay_per_distance"] = df["total_delay"] / (df["Flight Distance"] + 1.0)
        df["is_delayed"] = (df["total_delay"] > 0).astype(np.int8)

        # 2. Domain satisfaction indices
        df["inflight_comfort_index"] = df[["Seat comfort", "Inflight entertainment", "Cleanliness", "Food and drink"]].mean(axis=1)
        df["digital_exp_index"] = df[["Inflight wifi service", "Ease of Online booking"]].mean(axis=1)
        df["service_quality_index"] = df[["On-board service", "Leg room service", "Checkin service"]].mean(axis=1)
        df["ground_service_index"] = df[["Checkin service", "Gate location", "Departure/Arrival time convenient"]].mean(axis=1)

        # 3. Rating statistics & Asymmetric Pain Points
        s_slice = df[SURVEY_FEATURES]
        df["ratings_sum"] = s_slice.sum(axis=1)
        df["ratings_mean"] = s_slice.mean(axis=1)
        df["ratings_std"] = s_slice.std(axis=1).fillna(0.0)
        df["ratings_min"] = s_slice.min(axis=1)
        df["ratings_max"] = s_slice.max(axis=1)
        df["low_rating_count"] = (s_slice <= 2).sum(axis=1)
        df["high_rating_count"] = (s_slice >= 4).sum(axis=1)

        # 4. Cross categoricals
        df["class_travel_type"] = df["Class"].astype(str) + "_" + df["Type of Travel"].astype(str)
        df["customer_class"] = df["Customer Type"].astype(str) + "_" + df["Class"].astype(str)

        # 5. Route frequency mapping
        df["route_frequency"] = df["Flight Distance"].map(dist_freq)

    # Encode categoricals
    cat_cols = ["Gender", "Customer Type", "Type of Travel", "Class", "class_travel_type", "customer_class"]
    enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    tr[cat_cols] = enc.fit_transform(tr[cat_cols].astype(str))
    te[cat_cols] = enc.transform(te[cat_cols].astype(str))

    return tr, te

train_feat, test_feat = engineer_base_features(train_raw, test_raw)
print("Base engineering completed. Total columns:", train_feat.shape[1])


## 3. In-Fold Bayesian Smoothed Route-ID Target Encoding
Applying M-estimate Bayesian smoothing strictly within folds ($K=5$):
$$S_i = \frac{n_i \cdot \bar{y}_i + m \cdot \bar{y}_{global}}{n_i + m}$$
Smoothing factor $m=20$ prevents small sample route noise while unlocking the $+0.0017$ AUC route signal.


In [ ]:
def compute_route_target_encoding(train_df, test_df, folds, smoothing=20.0):
    train_encoded = train_df.copy()
    test_encoded = test_df.copy()

    train_encoded["route_te"] = np.nan
    test_te_folds = np.zeros((len(test_df), len(folds)))

    for fold_idx, (t_idx, v_idx) in enumerate(folds):
        tr_part = train_df.iloc[t_idx]
        val_part = train_df.iloc[v_idx]

        global_mean = tr_part["satisfaction"].mean()
        stats = tr_part.groupby("Flight Distance")["satisfaction"].agg(["count", "mean"])
        
        # Bayesian M-estimate
        smooth = (stats["count"] * stats["mean"] + smoothing * global_mean) / (stats["count"] + smoothing)
        te_map = smooth.to_dict()

        # Validation fold mapping
        train_encoded.iloc[v_idx, train_encoded.columns.get_loc("route_te")] = \
            val_part["Flight Distance"].map(te_map).fillna(global_mean)

        # Test fold mapping
        test_te_folds[:, fold_idx] = test_df["Flight Distance"].map(te_map).fillna(global_mean)

    test_encoded["route_te"] = test_te_folds.mean(axis=1)
    return train_encoded, test_encoded

N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
folds = list(skf.split(train_feat, train_feat["satisfaction"]))

train_final, test_final = compute_route_target_encoding(train_feat, test_feat, folds, smoothing=20.0)

feature_cols = [c for c in train_final.columns if c not in ["id", "satisfaction"]]
print(f"✅ Final feature set ready: {len(feature_cols)} features")


## 4. 5-Fold Stratified CV Tri-GBDT Modeling (LightGBM, CatBoost, XGBoost)
Training diverse gradient boosted architectures with identical fold splits and early stopping.


In [ ]:
X = train_final[feature_cols]
y = train_final["satisfaction"]
X_test = test_final[feature_cols]

oof_preds = {}
test_preds = {}

# 1. LightGBM
print("=" * 60)
print("🚀 [1/3] Training 5-Fold LightGBM...")
lgb_oof = np.zeros(len(X), dtype=np.float32)
lgb_test = np.zeros(len(X_test), dtype=np.float32)

for fold, (t_idx, v_idx) in enumerate(folds):
    model = lgb.LGBMClassifier(
        objective="binary", metric="auc", learning_rate=0.03, num_leaves=63,
        max_depth=-1, min_child_samples=50, subsample=0.8, colsample_bytree=0.8,
        n_estimators=2000, random_state=42, n_jobs=-1, verbose=-1
    )
    model.fit(
        X.iloc[t_idx], y.iloc[t_idx],
        eval_set=[(X.iloc[v_idx], y.iloc[v_idx])],
        callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=False)]
    )
    val_pred = model.predict_proba(X.iloc[v_idx])[:, 1]
    lgb_oof[v_idx] = val_pred
    lgb_test += model.predict_proba(X_test)[:, 1] / N_SPLITS
    print(f"  LGB Fold {fold + 1} ROC-AUC: {roc_auc_score(y.iloc[v_idx], val_pred):.5f}")

lgb_auc = roc_auc_score(y, lgb_oof)
print(f"⭐ LightGBM Overall OOF ROC-AUC: {lgb_auc:.5f}")
oof_preds["LGBM"] = lgb_oof
test_preds["LGBM"] = lgb_test

# 2. CatBoost
print("\n" + "=" * 60)
print("🚀 [2/3] Training 5-Fold CatBoost...")
cat_oof = np.zeros(len(X), dtype=np.float32)
cat_test = np.zeros(len(X_test), dtype=np.float32)

for fold, (t_idx, v_idx) in enumerate(folds):
    model = CatBoostClassifier(
        iterations=2000, learning_rate=0.04, depth=7, eval_metric="AUC",
        random_seed=42, thread_count=-1, verbose=False
    )
    model.fit(
        X.iloc[t_idx], y.iloc[t_idx],
        eval_set=(X.iloc[v_idx], y.iloc[v_idx]),
        early_stopping_rounds=100, verbose=False
    )
    val_pred = model.predict_proba(X.iloc[v_idx])[:, 1]
    cat_oof[v_idx] = val_pred
    cat_test += model.predict_proba(X_test)[:, 1] / N_SPLITS
    print(f"  CAT Fold {fold + 1} ROC-AUC: {roc_auc_score(y.iloc[v_idx], val_pred):.5f}")

cat_auc = roc_auc_score(y, cat_oof)
print(f"⭐ CatBoost Overall OOF ROC-AUC: {cat_auc:.5f}")
oof_preds["CatBoost"] = cat_oof
test_preds["CatBoost"] = cat_test

# 3. XGBoost
print("\n" + "=" * 60)
print("🚀 [3/3] Training 5-Fold XGBoost...")
xgb_oof = np.zeros(len(X), dtype=np.float32)
xgb_test = np.zeros(len(X_test), dtype=np.float32)

for fold, (t_idx, v_idx) in enumerate(folds):
    model = xgb.XGBClassifier(
        learning_rate=0.03, max_depth=7, subsample=0.8, colsample_bytree=0.8,
        eval_metric="auc", tree_method="hist", n_estimators=2000,
        early_stopping_rounds=100, random_state=42, n_jobs=-1
    )
    model.fit(
        X.iloc[t_idx], y.iloc[t_idx],
        eval_set=[(X.iloc[v_idx], y.iloc[v_idx])],
        verbose=False
    )
    val_pred = model.predict_proba(X.iloc[v_idx])[:, 1]
    xgb_oof[v_idx] = val_pred
    xgb_test += model.predict_proba(X_test)[:, 1] / N_SPLITS
    print(f"  XGB Fold {fold + 1} ROC-AUC: {roc_auc_score(y.iloc[v_idx], val_pred):.5f}")

xgb_auc = roc_auc_score(y, xgb_oof)
print(f"⭐ XGBoost Overall OOF ROC-AUC: {xgb_auc:.5f}")
oof_preds["XGBoost"] = xgb_oof
test_preds["XGBoost"] = xgb_test


## 5. Nelder-Mead Rank Percentile Ensembling & Submission
Calibrating models into uniform percentiles $(rank(p) - 0.5) / N$ and solving for optimal non-negative convex weights to eliminate scale mismatch and collinearity artifacts.


In [ ]:
def rank_norm(p):
    return (rankdata(p) - 0.5) / len(p)

model_names = ["LGBM", "CatBoost", "XGBoost"]
ranked_oofs = [rank_norm(oof_preds[m]) for m in model_names]

def loss(params):
    exp_p = np.exp(params - np.max(params))
    w = exp_p / np.sum(exp_p)
    blend = sum(wi * oi for wi, oi in zip(w, ranked_oofs))
    return -roc_auc_score(y, blend)

res = minimize(loss, np.zeros(len(model_names)), method="Nelder-Mead", options={"maxiter": 500})
exp_p = np.exp(res.x - np.max(res.x))
opt_w = exp_p / np.sum(exp_p)
blend_auc = -res.fun

print("=" * 60)
print(f"🏆 Optimal Nelder-Mead Rank Ensemble OOF ROC-AUC: {blend_auc:.5f}")
for m, w in zip(model_names, opt_w):
    print(f"  - {m} Weight: {w:.4f} (Individual OOF: {roc_auc_score(y, oof_preds[m]):.5f})")

# Generate test predictions
ranked_tests = [rank_norm(test_preds[m]) for m in model_names]
blend_test = sum(wi * ti for wi, ti in zip(opt_w, ranked_tests))

sub["satisfaction"] = blend_test
sub.to_csv("submission.csv", index=False)
print(f"\n✅ Successfully generated submission.csv (Rows: {len(sub):,}, Nulls: {sub.isnull().sum().sum()})")

metrics = {
    "lgbm_oof": float(lgb_auc),
    "catboost_oof": float(cat_auc),
    "xgboost_oof": float(xgb_auc),
    "ensemble_blend_oof": float(blend_auc),
    "weights": {m: float(w) for m, w in zip(model_names, opt_w)}
}
with open("metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print("Saved metrics.json successfully.")
